# Cold-start (Nour)

* Baseline: `recommender.baselines.description_similarity`, the mean cosine similarity between
  a cold product's text vector and the products the customer rated 4–5 in training.
* Frozen-model evaluation: `recommender.frozen` (`save_frozen` → `evaluate_frozen`).

**Status: diagnostic only.** Cold-validation numbers here are *not* a selection signal (rule 6),
and no test split is touched. The SALR/BCD rows use the selected configurations from
`salr.ipynb` / `bcd.ipynb` at the *provisional* `lambda = 1e-3`; they are retrained in memory and
**not frozen**. The final cold-start table is produced only after all four methods are frozen:

```python
from recommender.frozen import evaluate_frozen
names = [f"{m}_seed{s}" for m in ("sgd", "sam", "salr", "bcd") for s in (616, 617, 618)]
evaluate_frozen(names, ds, "cold", "test")
```

In [1]:
import numpy as np, pandas as pd
from recommender.baselines import description_similarity, popularity, random_scores
from recommender.bcd import BCDConfig, train_bcd
from recommender.data import load_dataset
from recommender.evaluate import evaluate_model, evaluate_ranking, make_evaluator, select_checkpoint
from recommender.salr import SALRConfig, train_salr

ds = load_dataset()
SEEDS = (616, 617, 618)
for split in ("validation",):
    print(f"cold/{split}: {len(ds.eval_users('cold', split))} customers, "
          f"{len(ds.catalog_ids['cold_' + split])} candidate products")

cold/validation: 119 customers, 40 candidate products


## Baselines on validation

In [2]:
rows = []
for task in ("cold", "ordinary"):
    sim = evaluate_ranking(description_similarity(ds), ds, task, "validation")
    rnd = [evaluate_ranking(random_scores(s), ds, task, "validation") for s in SEEDS]
    rows += [{"task": task, "scorer": "description similarity", **sim},
             {"task": task, "scorer": "random (mean of 3 seeds)", "recall": np.mean([r["recall"] for r in rnd]),
              "ndcg": np.mean([r["ndcg"] for r in rnd]), "n_users": rnd[0]["n_users"]}]
rows.append({"task": "ordinary", "scorer": "popularity",
             **evaluate_ranking(popularity(ds), ds, "ordinary", "validation")})
baselines = pd.DataFrame(rows)
baselines.round(4)

,task,scorer,recall,ndcg,n_users
0,cold,description similarity,0.3758,0.2010,119
1,cold,random (mean of 3 seeds),0.2866,0.1530,119
2,ordinary,description similarity,0.1322,0.0644,121
3,ordinary,random (mean of 3 seeds),0.0523,0.0240,121
4,ordinary,popularity,0.1983,0.0817,121


## SALR and BCD selected checkpoints on cold validation (diagnostic, provisional lambda)

Checkpoints are chosen by **ordinary**-validation NDCG@10 as always; the cold numbers are read
off the same parameters, with `q_i = b_i = 0` for cold products.

In [3]:
evaluate = make_evaluator(ds)
configs = {
    "salr": lambda s: train_salr(ds, s, SALRConfig(16, 0.05, 1e-3, 1e-3), 1_000_000, 20_000, evaluate),
    "bcd": lambda s: train_bcd(ds, s, BCDConfig(1e-3, 1e-3, inner_tol=1e-4, max_inner=1), 1_000_000, 20_000, evaluate),
}
model_rows = []
for name, train in configs.items():
    for seed in SEEDS:
        with np.errstate(all="ignore"):
            best = select_checkpoint(train(seed))
        cold = evaluate_model(best["params"], ds, "cold", "validation")
        model_rows.append({"model": name, "seed": seed, "ordinary_val_ndcg": best["val_ndcg"],
                           "cold_val_ndcg": cold["ndcg"], "cold_val_recall": cold["recall"]})
models = pd.DataFrame(model_rows)
models.round(4)

,model,seed,ordinary_val_ndcg,cold_val_ndcg,cold_val_recall
0,salr,616,0.1435,0.2626,0.4427
1,salr,617,0.1413,0.2633,0.4303
2,salr,618,0.1428,0.2563,0.4462
3,bcd,616,0.1376,0.2326,0.3865
4,bcd,617,0.1338,0.2544,0.4168
5,bcd,618,0.1316,0.2268,0.3894


In [4]:
summary = models.groupby("model").agg(ordinary_val_ndcg=("ordinary_val_ndcg", "mean"),
                                     cold_val_ndcg=("cold_val_ndcg", "mean"),
                                     cold_val_ndcg_sd=("cold_val_ndcg", "std"),
                                     cold_val_recall=("cold_val_recall", "mean")).reset_index()
cold_base = baselines[baselines.task == "cold"].rename(columns={"scorer": "model", "ndcg": "cold_val_ndcg",
                                                              "recall": "cold_val_recall"})
table = pd.concat([summary, cold_base[["model", "cold_val_ndcg", "cold_val_recall"]]], ignore_index=True)
table.to_csv("results/coldstart_validation_diagnostic.csv", index=False)
table.round(4)

,model,ordinary_val_ndcg,cold_val_ndcg,cold_val_ndcg_sd,cold_val_recall
0,bcd,0.1343,0.2379,0.0145,0.3976
1,salr,0.1425,0.2607,0.0038,0.4397
2,description similarity,NaN,0.2010,NaN,0.3758
3,random (mean of 3 seeds),NaN,0.1530,NaN,0.2866
